# 第 08 章练习

完成下面的练习，结合自己的运行结果作答。


<a id="s-08-03_chapter_test"></a>


## 8.3 章节实践

已完成 8.2 节时，“运行 memcheck”和“两种工程对照”可直接使用已保存的诊断日志，跳过对应运行格。单独做练习时再运行这些格；“修复与记录”中的 baseline 对照仍需执行。

### 一、客观题

1. memcheck、racecheck、initcheck、synccheck 分别对应哪类故障？
2. 独立 ASC 与标准算子工程的 sanitizer 插桩位置分别是什么？
3. 故障程序返回非零是否等于实验失败？还必须检查什么？
4. `/dev/drv_debug` 不可读写时，msDebug 应标记 PASS、FAIL 还是 BLOCKED？
5. 判断：课程可以用 `sudo chmod` 临时开放 `/dev/drv_debug`。

### 二、运行 memcheck

运行已知的越界故障，从日志中找到越界操作及源码行号。说明工具为什么报错，并记录实际退出码。


In [ ]:
%%bash
set -e
cmake -S src -B build/standalone-memcheck -DLAB05_FAULT_MODE=memcheck -DLAB05_ENABLE_SANITIZER=ON
cmake --build build/standalone-memcheck -j
set +e
BUILD=build/standalone-memcheck
mssanitizer --tool=memcheck "$BUILD/lab05_add_sanitizer" 2>&1 | tee "$BUILD/mssanitizer.log"
rc=${PIPESTATUS[0]}
echo "工具退出码：$rc"
set -e
grep -Ei 'illegal (read|write)|out[- ]of[- ]bounds' "$BUILD/mssanitizer.log"
echo 'SANITIZER_RESULT status=EXPECTED_DIAGNOSTIC mode=memcheck'


### 三、两种工程对照

学生修改：从 racecheck、initcheck、synccheck 中任选一种，替换下面的 `MODE`。用同一模式运行两种工程，比较诊断并说明插桩位置为什么不同。


In [ ]:
%%bash
set -e
# 学生修改：本次要分析的故障模式。
MODE=racecheck
BUILD=build/standalone-$MODE
cmake -S src -B "$BUILD" -DLAB05_FAULT_MODE="$MODE" -DLAB05_ENABLE_SANITIZER=ON
cmake --build "$BUILD" -j
set +e
mssanitizer --tool="$MODE" "$BUILD/lab05_add_sanitizer" 2>&1 | tee "$BUILD/mssanitizer.log"
rc=${PIPESTATUS[0]}
set -e
echo "工具退出码：$rc"
case "$MODE" in
  racecheck) grep -Ei 'RAW hazard|data race' "$BUILD/mssanitizer.log" ;;
  initcheck) grep -Ei 'uninitialized' "$BUILD/mssanitizer.log" ;;
  synccheck) grep -Ei 'unpaired[ _-]*set_flag' "$BUILD/mssanitizer.log" ;;
esac
bash src/run_sanitizer.sh operator "$MODE"


以 racecheck 为例，两份日志分别位于 `build/standalone-racecheck/mssanitizer.log` 和 `src/operator_project_sanitizer/build-racecheck/mssanitizer.log`。找到各自的 RAW hazard 诊断和 Kernel 源码位置。


### 四、修复与记录

阅读 `src/add_sanitizer.asc`，说明 memcheck 与 baseline 分支的数据搬运长度有何不同。
下面切回 baseline，保持 memcheck 插桩开启。运行后应正常退出、输出 `test pass`，且越界诊断消失。


In [ ]:
%%bash
set -e
BUILD=build/repaired-memcheck
cmake -S src -B "$BUILD" -DLAB05_FAULT_MODE=baseline -DLAB05_ENABLE_SANITIZER=ON
cmake --build "$BUILD" -j
set +e
mssanitizer --tool=memcheck "$BUILD/lab05_add_sanitizer" 2>&1 | tee "$BUILD/mssanitizer.log"
rc=${PIPESTATUS[0]}
set -e
test "$rc" -eq 0
if grep -Ei 'uninitialized|illegal (read|write)|out[- ]of[- ]bounds|RAW hazard|unpaired[ _-]*set_flag' "$BUILD/mssanitizer.log"; then
  exit 1
fi
grep -i 'test pass' "$BUILD/mssanitizer.log"
echo 'SANITIZER_RESULT status=PASS mode=memcheck'


### 整理观察结果

在这里写下每种故障的关键诊断、源码行号和原因，再说明修复后的变化。

msDebug 记录是否命中断点；平台条件不足时写 `BLOCKED` 及原因。


### 完成标准

能根据诊断定位源码，完成两种工程的对照，并确认修复后数值正确、原诊断消失。


完成本节练习后查看：[参考答案](answer/08.03_answer.md)